In [ ]:
# Cell 1
%pip install pandas==2.2.3 numpy==2.3.5 scikit-learn==1.8.0 matplotlib==3.10.8

In [ ]:
# Cell 2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# Cell 3 
weather = pd.read_csv("multicity_weather.csv")
for column in ["forecast_time_utc", "event_end_utc"]:
    weather[column] = pd.to_datetime(weather[column], utc=True)
weather["heat_event"] = (weather["observed_max_temperature_c"] >= 35).astype(int)
weather["heavy_rain_event"] = (weather["observed_rainfall_mm"] >= 50).astype(int)
weather["extreme_weather"] = weather[["heat_event", "heavy_rain_event"]].max(axis=1)

weather.groupby("city").size().rename("Available days")

In [ ]:
# Cell 4 
weather["month_sin"] = np.sin(2 * np.pi * weather["month"] / 12)
weather["month_cos"] = np.cos(2 * np.pi * weather["month"] / 12)
weather["log_forecast_rain"] = np.log1p(weather["forecast_rainfall_mm"])
weather["log_previous_rain"] = np.log1p(weather["previous_daily_rainfall_mm"])
weather["dew_point_spread"] = weather["temperature_c"] - weather["dew_point_c"]
weather["forecast_temperature_change"] = weather["forecast_max_temperature_c"] - weather["temperature_c"]
city_columns, interactions = [], []
for city in ["Sydney", "Tokyo", "Hong Kong", "Madrid", "Auckland"]:
    column = "city_" + city.lower().replace(" ", "_")
    weather[column] = (weather["city"] == city).astype(int)
    city_columns.append(column)
    for feature in ["forecast_max_temperature_c", "log_forecast_rain"]:
        name = feature + "_" + column
        weather[name] = weather[feature] * weather[column]
        interactions.append(name)

forest_inputs = [
    "forecast_max_temperature_c", "forecast_average_humidity_percent",
    "forecast_lowest_pressure_hpa", "log_forecast_rain", "month_sin", "month_cos"
] + city_columns + [
    "forecast_temperature_range_c", "forecast_max_hourly_rain_mm",
    "forecast_max_6h_rain_mm", "forecast_wet_hours", "forecast_pressure_change_24h_hpa"
]
logistic_inputs = [
    "forecast_max_temperature_c", "forecast_average_humidity_percent",
    "forecast_lowest_pressure_hpa", "temperature_c", "dew_point_c", "air_pressure_hpa",
    "month_sin", "month_cos", "log_forecast_rain", "log_previous_rain",
    "dew_point_spread", "forecast_temperature_change"
] + city_columns + interactions
inputs = {"Random Forest": forest_inputs, "Logistic Regression": logistic_inputs}

In [ ]:
# Cell 5 
training = weather[weather["event_end_utc"] < "2024-01-01"].copy()
validation = weather[(weather["forecast_time_utc"] >= "2024-01-01") &
                     (weather["event_end_utc"] < "2025-01-01")].copy()
testing = weather[weather["evaluation_status"] == "previously_unused_four_city_holdout"].copy()
sydney = weather[(weather["city"] == "Sydney") &
                 (weather["forecast_time_utc"] >= "2025-01-01")].copy()
# New four-city test runs are 6h old; development runs are 22–24h old.
# Sydney 2025 was viewed earlier, so its results remain exploratory.
for name, group in [("Training", training), ("Validation", validation),
                    ("Four-city test", testing), ("Sydney exploratory", sydney)]:
    print(name, ":", len(group), "days")

In [ ]:
# Cell 6
models = {
    "Random Forest": make_pipeline(SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=2,
                               max_features=0.5, random_state=42, n_jobs=-1)),
    "Logistic Regression": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
        LogisticRegression(C=1.0, max_iter=2000))
}
for name, model in models.items():
    model.fit(training[inputs[name]], training["extreme_weather"])


In [ ]:
# Cell 7 
def measure(name, answers, predicted):
    tn, fp, fn, tp = confusion_matrix(answers, predicted, labels=[0, 1]).ravel()
    precision = tp / (tp + fp) if tp + fp else np.nan
    recall = tp / (tp + fn) if tp + fn else np.nan
    return {"Model": name, "Days": len(answers), "Actual extremes": int(np.sum(answers)),
            "Precision (%)": 100 * precision, "Recall (%)": 100 * recall,
            "Correct warnings": tp, "False alarms": fp, "Missed extremes": fn,
            "Correct normal": tn, "Both >=70%": precision >= 0.70 and recall >= 0.70}

thresholds, validation_rows = {}, []
for name, model in models.items():
    scores = model.predict_proba(validation[inputs[name]])[:, 1]
    precision, recall, cutoffs = precision_recall_curve(validation["extreme_weather"], scores)
    choices = pd.DataFrame({"Threshold": cutoffs, "Precision": precision[:-1], "Recall": recall[:-1]})
    choices["Weaker metric"] = choices[["Precision", "Recall"]].min(axis=1)
    choices["F1"] = 2 * choices.Precision * choices.Recall / (choices.Precision + choices.Recall + 1e-12)
    chosen = choices.sort_values(["Weaker metric", "F1", "Precision"], ascending=False).iloc[0]
    thresholds[name] = chosen.Threshold
    validation_rows.append({**measure(name, validation["extreme_weather"], scores >= chosen.Threshold),
                            "Threshold": chosen.Threshold})
validation_results = pd.DataFrame(validation_rows)
validation_results.round(3)

In [ ]:
# Cell 8 
predictions, score_tables, result_rows = {}, {}, []
groups = {"Four-city holdout": testing, "Sydney exploratory": sydney}
for group_name, group in groups.items():
    table = group[["city", "forecast_time_utc", "heat_event", "heavy_rain_event", "extreme_weather"]].copy()
    score_tables[group_name] = {}
    for name, model in models.items():
        scores = model.predict_proba(group[inputs[name]])[:, 1]
        score_tables[group_name][name] = scores
        table[name] = scores >= thresholds[name]
    table["Direct forecast"] = (group["forecast_max_temperature_c"] >= 35) | (group["forecast_rainfall_mm"] >= 50)
    for name in [*models, "Direct forecast"]:
        result_rows.append({"Evaluation": group_name, **measure(name, table["extreme_weather"], table[name])})
    predictions[group_name] = table
results = pd.DataFrame(result_rows)
results.round(2)

In [ ]:
# Cell 9 
city_rows, hazard_rows = [], []
for group_name, table in predictions.items():
    for city, local in table.groupby("city"):
        for name in [*models, "Direct forecast"]:
            city_rows.append({"City": city, "Evaluation": group_name,
                              **measure(name, local["extreme_weather"], local[name])})
    for name in [*models, "Direct forecast"]:
        for hazard in ["heat_event", "heavy_rain_event"]:
            actual = table[hazard] == 1
            detected = int((actual & table[name]).sum())
            hazard_rows.append({"Evaluation": group_name, "Model": name, "Hazard": hazard,
                               "Actual": int(actual.sum()), "Detected": detected,
                               "Recall (%)": 100 * detected / actual.sum() if actual.sum() else np.nan})
city_results = pd.DataFrame(city_rows)
hazard_results = pd.DataFrame(hazard_rows)
display(city_results.round(2))
display(hazard_results.round(2))

In [ ]:
# Cell 10 
table = predictions["Four-city holdout"]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
fig.suptitle("Four-city 2025 holdout")
for axis, name in zip(axes, [*models, "Direct forecast"]):
    ConfusionMatrixDisplay.from_predictions(table["extreme_weather"], table[name],
        labels=[0, 1], display_labels=["Normal", "Extreme"], ax=axis, colorbar=False, cmap="Blues")
    axis.set_title(name)
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=200)
plt.show()

In [ ]:
# Cell 11 
plt.figure(figsize=(7, 5))
plt.title("Four-city 2025 holdout")
for name, scores in score_tables["Four-city holdout"].items():
    precision, recall, _ = precision_recall_curve(table["extreme_weather"], scores)
    plt.step(recall[:-1], precision[:-1], where="post", label=name)
    row = results[(results.Evaluation == "Four-city holdout") & (results.Model == name)].iloc[0]
    plt.scatter(row["Recall (%)"] / 100, row["Precision (%)"] / 100, s=45)
plt.axhline(0.70, color="grey", linestyle="--")
plt.axvline(0.70, color="grey", linestyle="--")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.legend()
plt.tight_layout()
plt.savefig("precision_recall.png", dpi=200)
plt.show()

In [ ]:
# Cell 12 
validation_results.to_csv("validation_results.csv", index=False)
results.to_csv("comparison_results.csv", index=False)
city_results.to_csv("city_results.csv", index=False)
hazard_results.to_csv("hazard_results.csv", index=False)
pd.concat(predictions, names=["Evaluation"]).reset_index(level=0).to_csv("test_predictions.csv", index=False)
print("Saved five result tables and two charts.")